# Free MuseTalk 1.5 lip-sync test (Kaggle GPU)

This notebook runs the open-source MuseTalk 1.5 model on Kaggle's free GPU runtime. It does not call a paid inference API and does not publish anything to YouTube.

## Before running
1. In Kaggle, choose **Settings → Accelerator → GPU** and enable Internet.
2. Create a **private Kaggle Dataset** containing exactly one test presenter `.mp4` and its matching narration `.wav` (or `.mp3`). Add that dataset as this notebook's input. Do not upload API keys or credentials.
3. Use a short 4–8 second clip for the first test. Prefer the existing Kling presenter footage and the exact narration excerpt intended for that shot.
4. Run cells in order. The first setup/model-download run may take time and consumes Kaggle's free session quota, not money.

If a GPU is unavailable or the session quota is exhausted, stop and retry later. Do not switch to any paid service.

In [ ]:
import os, subprocess, sys, pathlib, glob, json, textwrap
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No Kaggle GPU is attached. Open Notebook Settings, select GPU, then restart the session.')
print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_properties(0).total_memory >= 4 * 1024**3, 'GPU has less than 4 GB VRAM; stop rather than use a paid service.'

In [ ]:
%%bash
%%bash
set -e
apt-get update -qq && apt-get install -y -qq ffmpeg
git clone https://github.com/TMElyralab/MuseTalk.git /kaggle/working/MuseTalk
cd /kaggle/working/MuseTalk
python -m pip install -q --upgrade pip
python -m pip install -q -r requirements.txt
python -m pip install -q huggingface_hub==0.30.2 gdown
echo 'MuseTalk source and Python dependencies installed.'


In [ ]:
%%bash
%%bash
set -e
cd /kaggle/working/MuseTalk
bash download_weights.sh
echo 'Checkpoint download finished.'


In [ ]:
# Find one video and one matching audio file in the attached private Kaggle Dataset.
from pathlib import Path
import os, glob, shutil
video_candidates = [p for p in glob.glob('/kaggle/input/**/*', recursive=True) if p.lower().endswith(('.mp4','.mov','.mkv'))]
audio_candidates = [p for p in glob.glob('/kaggle/input/**/*', recursive=True) if p.lower().endswith(('.wav','.mp3','.m4a'))]
print('Video files:', video_candidates)
print('Audio files:', audio_candidates)
if len(video_candidates) != 1 or len(audio_candidates) != 1:
    raise RuntimeError('Attach a private Kaggle Dataset containing exactly one test video and one matching narration audio file.')
work = Path('/kaggle/working/MuseTalk')
video_path = Path(video_candidates[0])
audio_path = Path(audio_candidates[0])
data_dir = work / 'data' / 'free_test'
data_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(video_path, data_dir / 'presenter.mp4')
shutil.copy2(audio_path, data_dir / ('narration' + audio_path.suffix))
print('Input staged:', list(data_dir.iterdir()))

In [ ]:
# Configure one inference task. MuseTalk recommends 25 fps input for best results.
import yaml
config = {
  'task_0': {
    'video_path': str(data_dir / 'presenter.mp4'),
    'audio_path': str(data_dir / ('narration' + audio_path.suffix)),
    'bbox_shift': 0
  }
}
config_path = work / 'configs' / 'inference' / 'kaggle_free_test.yaml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.safe_dump(config, f, sort_keys=False)
print(config_path.read_text())

In [ ]:
%%bash
%%bash
set -e
cd /kaggle/working/MuseTalk
python -m scripts.inference \
  --inference_config configs/inference/kaggle_free_test.yaml \
  --result_dir /kaggle/working/musetalk_preview \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 \
  --fps 25 \
  --batch_size 4 \
  --use_float16


In [ ]:
# Locate the rendered video and check its duration. Download it from the Kaggle output panel.
from pathlib import Path
import subprocess
outputs = list(Path('/kaggle/working/musetalk_preview').rglob('*.mp4'))
if not outputs:
    raise RuntimeError('MuseTalk did not produce an MP4. Inspect the inference cell logs; do not publish an unsynchronized fallback.')
for p in outputs:
    probe = subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)], text=True).strip()
    print({'file': str(p), 'bytes': p.stat().st_size, 'duration_seconds': float(probe)})
    display(__import__('IPython').display.Video(str(p), embed=True))